In [ ]:
# Cleanup: Shutdown APM gracefully
if apm_manager.is_initialized():
    print("Shutting down APM...")
    apm_manager.shutdown()
    print("✅ APM shut down gracefully")
else:
    print("⚠️  APM was not initialized (no keys)")

print("\\n" + "="*60)
print("Module 7.2 Complete!")
print("="*60)
print("\\nNext Steps:")
print("1. Sign up for Datadog (14-day free trial)")
print("2. Add DD_API_KEY to .env")
print("3. Run FastAPI app with: ddtrace-run uvicorn app:app")
print("4. View APM data in Datadog UI")
print("\\nNext Module: M7.3 - Cost Optimization for Observability")

## Section 10: Production Deployment

### Production Deployment Checklist

Before deploying APM to production:

- [ ] Sample rate ≤10% (`DD_TRACE_SAMPLE_RATE=0.1`)
- [ ] Profiling capture ≤1% (`DD_PROFILING_CAPTURE_PCT=1`)
- [ ] Max CPU overhead limit set to 5%
- [ ] Cost alerting configured
- [ ] Excluded paths configured (`/health`, `/metrics`)
- [ ] Load tested with APM enabled
- [ ] Rollback plan ready

### Deployment Command

Instead of:
```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

Use the `ddtrace-run` wrapper:
```bash
ddtrace-run uvicorn app:app --host 0.0.0.0 --port 8000
```

### Emergency Rollback

```bash
# Disable APM without restart
export DD_PROFILING_ENABLED=false
export DD_TRACE_SAMPLE_RATE=0.0

# Or restart without ddtrace-run wrapper
uvicorn app:app --host 0.0.0.0 --port 8000
```

### Scaling Configurations

**Small (1K req/hour):** 10% sampling, 1% profiling  \n**Medium (10K req/hour):** 5% sampling, 1% profiling  \n**Large (100K+ req/hour):** 1% sampling, 0.5% profiling or disable

In [ ]:
# Decision Matrix: Should you use APM?
print("Decision Matrix: Use APM?\\n")

criteria = {
    "Traffic >1K req/hour": True,
    "Known perf problem (P95 >3s)": True,
    "Budget $50-200/month": True,
    "Team size 3+": True,
    "No data sovereignty restrictions": True
}

use_apm = all(criteria.values())

for criterion, met in criteria.items():
    status = "✅" if met else "❌"
    print(f"{status} {criterion}")

print(f"\\nRecommendation: {'USE Datadog APM' if use_apm else 'Consider alternatives'}")

# Expected:
# ✅ Traffic >1K req/hour
# ✅ Known perf problem (P95 >3s)
# ✅ Budget $50-200/month
# ✅ Team size 3+
# Recommendation: USE Datadog APM

## Section 9: Decision Card

### ✅ BENEFIT
Deep code-level profiling reveals bottlenecks down to specific function calls and line numbers. Reduces debugging time from hours to minutes by showing CPU hotspots, memory leaks, and slow queries with flame graphs. Correlates performance issues with traces from M7.1.

### ❌ LIMITATION
Adds 2-5% CPU overhead in production even with conservative sampling (1% profiling, 10% trace sampling). Cost scales rapidly: $51/month minimum, rising to $300+/month at 100K requests/hour due to per-span analysis fees ($5 per 1M spans). Memory profiling shows allocations but struggles to detect slow retention-based leaks.

### 💰 COST
- **Time:** 2-4 hours initial setup, 1-2 days production tuning
- **Monthly:** $51-100 small (1-3 hosts), $300-800 medium (10-15 hosts, 10M spans/day)
- **Complexity:** 300+ lines APM config code, understanding profiling overhead vs visibility trade-offs

### 🤔 USE WHEN
Traffic exceeds 1K req/hour with known performance problems (P95 >3s), budget allows $50-200/month for APM, team of 3+ engineers who will actively monitor dashboards, and no compliance restrictions on sending telemetry to third-party services.

### 🚫 AVOID WHEN
Traffic below 1K req/hour (insufficient data - use py-spy), budget under $100/month total (APM would be 50%+ of costs - use Grafana Tempo), processing sensitive data requiring full sovereignty (use self-hosted), or no known performance issues yet (premature optimization - wait until P95 crosses 3s).

In [ ]:
# Load cost estimates from example data
with open("example_data.json") as f:
    data = json.load(f)

print("APM Cost Comparison:")
print("\nDatadog APM:")
for scale, info in data["cost_estimates"].items():
    print(f"  {scale}: ${info['monthly_cost_usd']}/mo - {info['description']}")

print("\nOpen-Source (Tempo + Pyroscope):")
print("  Small: $20/mo - Self-hosted on 1 small VM")
print("  Medium: $40/mo - 2 VMs with load balancing")

# Expected:
# Small: $51/mo - 1 host, 1K req/hour
# Medium: $120/mo - 3-5 hosts, 10K req/hour
# Large: $800/mo - 15+ hosts, 100K req/hour

## Section 8: Alternative Solutions

### Option 1: Open-Source APM (Grafana Tempo + Pyroscope)

**Pros:**
- $20-40/month (vs $51-300 for Datadog)
- Full data sovereignty
- Self-hosted control

**Cons:**
- 2-3 hours setup time (vs 30 min)
- Requires DevOps expertise
- Manual integration work

**When to use:** Team size 5+, budget <$100/month, data sovereignty required

### Option 2: Cloud Provider APM (AWS X-Ray, GCP Cloud Profiler)

**Pros:**
- $15-50/month
- Integrated with cloud platform
- Minimal setup

**Cons:**
- Vendor lock-in
- Basic profiling only
- Limited flame graphs

**When to use:** Already on AWS/GCP, budget-conscious, simple needs

### Option 3: Manual Profiling with py-spy

**Pros:**
- $0 cost
- 5-minute setup
- Zero overhead when not profiling

**Cons:**
- Manual process (not continuous)
- No production correlation with traces
- Must reproduce issue locally

**When to use:** Traffic <1K req/hour, one-off debugging

In [ ]:
# Demonstrate config validation (production safety)
print("Testing config validation...")

# These values trigger warnings (shown during config initialization)
print(f"Sample Rate: {apm_config.DD_TRACE_SAMPLE_RATE}")
print(f"Profiling Capture: {apm_config.DD_PROFILING_CAPTURE_PCT}%")

# Unsafe values would raise ValueError in production
# Example: DD_TRACE_SAMPLE_RATE=0.8 in DD_ENV=production → ValueError

print("\n✅ Production safety checks passed")

# Expected:
# Sample Rate: 0.1
# Profiling Capture: 1%
# ✅ Production safety checks passed

## Section 7: Common Failures & How to Fix Them

### Failure 1: APM Overhead Crushing Performance (5-15% slowdown)

**Symptoms:**
- P95 latency increases from 800ms to 1.2s (50% slowdown)
- CPU usage spikes to 95%
- APM warnings: "High profiling overhead: 15.2% CPU"

**Root Cause:** Aggressive profiling (10% capture, 100% sampling)

**Fix:**
```python
DD_PROFILING_CAPTURE_PCT=1  # Not 10
DD_TRACE_SAMPLE_RATE=0.1    # Not 1.0
DD_PROFILING_MAX_TIME_USAGE_PCT=5
```

### Failure 2: Memory Profiler Crashes Production

**Symptoms:** MemoryError, OOM killed, hangs

**Root Cause:** Using `@profile` decorator in production (20GB overhead)

**Fix:** NEVER use `@profile` in production. Use tracemalloc instead.

### Failure 3: APM Cost Explosion ($500+ bill)

**Symptoms:** Bill shows $460/month vs expected $51

**Root Cause:** Developer set `DD_TRACE_SAMPLE_RATE=1.0` for debugging

**Fix:** Add config validation that fails deployment with unsafe settings

In [ ]:
# Create LRU cache with size limit
cache = QueryCache(max_size=10)

# Add 15 items (will trigger eviction)
print("Adding 15 items to cache (max_size=10)...")
for i in range(15):
    cache.set(f"query_{i}", "user_1", {"result": f"answer_{i}"})

print(f"Cache size: {len(cache._cache)} (should be 10)")

# Recent items should be in cache
hit = cache.get("query_14", "user_1")
print(f"query_14 in cache: {hit is not None}")

# Old items should be evicted
miss = cache.get("query_0", "user_1")
print(f"query_0 in cache: {miss is not None} (should be False)")

# Expected:
# Cache size: 10 (should be 10)
# query_14 in cache: True
# query_0 in cache: False

## Section 6: Query Cache with LRU Eviction (Leak Fix)

### The Fix

Implement LRU cache with `max_size` limit to prevent unbounded growth.

**Before (LEAK):**
```python
cache[f"{user}:{query}:{time.time()}"] = result  # New key every time
```

**After (FIXED):**
```python
cache[f"{user}:{hash(query)}"] = result
if len(cache) > max_size:
    cache.popitem(last=False)  # Remove oldest
```

### Benefits
- Predictable memory usage
- O(1) lookups
- Automatic eviction of oldest entries

In [ ]:
# Initialize memory profiler
profiler = MemoryProfiler()

# Get baseline memory
baseline = profiler.get_memory_stats()
print("Baseline memory:")
print(f"  Current: {baseline['current_mb']:.2f} MB")
print(f"  Peak: {baseline['peak_mb']:.2f} MB")

# Simulate memory leak
print("\nSimulating memory leak (3 iterations)...")
for i in range(3):
    profiler.cache_documents_with_leak(["Sample document text"] * 100)

# Check memory after leak
after_leak = profiler.get_memory_stats()
print(f"\nAfter leak:")
print(f"  Current: {after_leak['current_mb']:.2f} MB")
print(f"  Growth: {after_leak['growth_mb']:.2f} MB")

# Expected:
# Baseline: ~5-10 MB
# After leak: ~20-30 MB
# Growth: ~15-20 MB

## Section 5: Memory Profiling & Leak Detection

### The Challenge

APM shows memory ALLOCATION but not RETENTION. This is a key limitation.

**Example Leak Pattern:**
```python
# LEAK: Cache grows unbounded
cache[f"query_{time.time()}"] = result  # Unique key every time!
```

### Detection Strategy

1. **tracemalloc** - Production-safe memory tracking (built into Python)
2. **objgraph** - Find objects that aren't being freed (development only)
3. **APM flame graphs** - Show allocation hotspots (but not retention)

### The Fix: LRU Cache with Eviction

Use `OrderedDict` with max_size to prevent unbounded growth.

In [ ]:
# Create profiled RAG pipeline
pipeline = ProfiledRAGPipeline(use_apm=apm_manager.is_initialized())

# Process a test query
import time
start = time.time()

result = pipeline.process_query(
    query="What are GDPR compliance requirements for data retention?",
    user_id="user_123"
)

duration = time.time() - start

print(f"Query completed in {duration:.2f}s")
print(f"Response: {result['response'][:60]}...")
print(f"Results: {result['num_results']} documents")

# Expected:
# Query completed in ~1.5s
# Response: Based on the context, here is the answer to 'What are...
# Results: 10 documents

## Section 4: Profiled RAG Pipeline

### Custom Profiling Annotations

We add `@tracer.wrap` decorators to RAG pipeline functions so APM knows which parts matter most.

**What Gets Profiled:**
- `rag.query` - Main query processing
- `rag.embed_query` - Embedding generation (usually fast ~200ms)
- `rag.search_vector_db` - Network call to Pinecone (~300ms)
- `rag.process_context` - **THIS IS WHERE BOTTLENECKS HIDE** (~2.5s)
- `rag.remove_overlaps` - O(n²) comparison (intentional bottleneck)

### Custom Tags

APM spans include custom tags visible in Datadog UI:
- `user.id` - Which user made the request
- `query.length` - Query size
- `results.count` - Number of documents returned

In [ ]:
# Initialize APM Manager
apm_manager = APMManager(apm_config)

if apm_config.is_enabled():
    print("Initializing APM...")
    success = apm_manager.initialize()
    if success:
        print("✅ APM initialized successfully")
    else:
        print("❌ APM initialization failed")
else:
    print("⚠️  Skipping APM initialization (no DD_API_KEY)")
    print("   Pipeline will run without APM profiling")

# Expected:
# ⚠️  Skipping APM initialization (no DD_API_KEY)
#    Pipeline will run without APM profiling
# (or ✅ if you have DD_API_KEY configured)

## Section 3: Initialize APM Manager

### Step 1: Configure Datadog APM Integration

We integrate Datadog with your M7.1 OpenTelemetry setup WITHOUT double instrumentation.

**Key Configuration Values:**
- **1% profiling capture:** Production-safe, <1% overhead
- **10% trace sampling:** Balances cost ($5/1M spans) with visibility
- **5% max CPU overhead:** Safety limit - APM won't exceed 5% CPU
- **50% DB query sampling:** Good signal without excessive cost

### OpenTelemetry Bridge

The bridge makes Datadog understand your existing OTel spans from M7.1 automatically.

In [ ]:
# Verify APM configuration with production-safe defaults
print("APM Configuration:")
print(f"  Service: {apm_config.DD_SERVICE}")
print(f"  Environment: {apm_config.DD_ENV}")
print(f"  Profiling Capture: {apm_config.DD_PROFILING_CAPTURE_PCT}%")
print(f"  Trace Sample Rate: {apm_config.DD_TRACE_SAMPLE_RATE * 100}%")
print(f"  Max CPU Overhead: {apm_config.DD_PROFILING_MAX_TIME_USAGE_PCT}%")

# Expected:
# APM Configuration:
#   Service: compliance-copilot-rag
#   Profiling Capture: 1%
#   Trace Sample Rate: 10.0%
#   Max CPU Overhead: 5%

## Section 2: Theory Foundation - APM vs Tracing

### How APM Works

**Analogy:** Debugging a traffic jam
- **Metrics** (Prometheus): '1,000 cars/hour, avg speed 20mph'
- **Tracing** (OpenTelemetry): 'Car #47 took 45 min, passing zones A → B → C'
- **APM**: 'Car #47 spent 30 min in zone B because left lane blocked at mile 23.7'

### APM Architecture
```
Your Python App
├── OpenTelemetry (Traces)
│   └── Span: "process_query" - 2.5s
│
└── Datadog APM (Profiling)
    └── WITHIN that span:
        ├── Function: embedding_model() - 200ms
        ├── Function: chunk_filter() - 2.1s ⚠️
        │   └── Line 187: nested loop - 1.8s ⚠️⚠️
        └── Function: format_response() - 200ms
```

### How It Works
1. APM agent samples your Python process (100 samples/sec)
2. Captures call stack (which functions executing)
3. Statistical profile: "85% of time in chunk_filter()"
4. Correlates with OpenTelemetry traces

**Key Point:** APM COMPLEMENTS tracing, doesn't replace it.

# Module 7.2: Application Performance Monitoring

**Duration:** 38 minutes  
**Prerequisites:** Level 1 M2.3 (Monitoring), Level 2 M7.1 (OpenTelemetry Tracing)

## Overview

This module adds **deep code-level profiling** to complement your existing tracing from M7.1.

**The Problem:** Traces show WHICH span is slow. APM shows WHY it's slow (which function, which line of code).

**Today's Focus:**
- Integrate Datadog APM with OpenTelemetry (no double instrumentation)
- Profile production code with <5% overhead
- Detect memory leaks and CPU hotspots
- Optimize database queries using APM analysis

## Section 1: Introduction & Setup

### The Gap We're Filling

Your M7.1 tracing shows:
```
Span: "process_context" - Duration: 2,580ms ⚠️ SLOW
```

But WHY is it slow?
- Slow loop?
- Memory allocation?
- Regex operation?

**APM answers:** `chunk_overlap_filter()` at line 187 consuming 2.1s in O(n²) comparisons.

### Observability Pyramid
```
├── Logs (WHAT happened) ← Level 1 M2.3
├── Metrics (HOW MUCH) ← Level 1 M2.3
├── Traces (WHERE in pipeline) ← Level 2 M7.1
└── APM (WHY at code level) ← Today
```

In [ ]:
# Import core modules
import sys
import json
from pathlib import Path

# Import our APM components
from config import apm_config
from l2_m7_application_performance_monitoring import (
    APMManager,
    ProfiledRAGPipeline,
    MemoryProfiler,
    QueryCache
)

print("✅ Imports successful")
print(f"APM Service: {apm_config.DD_SERVICE}")
print(f"APM Enabled: {apm_config.is_enabled()}")

# Expected:
# ✅ Imports successful
# APM Service: compliance-copilot-rag
# APM Enabled: True (or False if no DD_API_KEY)